In [ ]:
#importing the necessary libraries
import pandas as pd
import numpy as np

In [ ]:
#step 1: load full CSV with multiple minerals
df_all = pd.read_csv("C:\\Users\\PC Galena\\Downloads\\MineralTDMeasured.csv", encoding='ISO-8859-1')

#step 2: filter for particular mineral (e.g., actinolite)
df_actinolite = df_all[df_all["mineral_name"].str.strip().str.lower() == "actinolite"].copy()

print(f"Original actinolite samples: {len(df_actinolite)}")

#step 3: identify composition (oxide) columns
metadata_cols = ["mineral_name", 'mineral_frequency',
 'sample_label',
 'rock_name',
 'classification',
 'latitude',
 'longitude',
 'doi/ref',
 'igsn',
 'analytical_method',
 'data_source']
composition_cols = [col for col in df_actinolite.columns if col not in metadata_cols]

#ensure numeric values
for col in composition_cols:
    df_actinolite[col] = pd.to_numeric(df_actinolite[col], errors='coerce')

#step 4: replicate to reach at least e.g., 4158 samples
target_samples = 4158
replication_factor = int(np.ceil(target_samples / len(df_actinolite)))
df_expanded = pd.concat([df_actinolite] * replication_factor, ignore_index=True).iloc[:target_samples].copy()

#step 5: apply random perturbation and scaling
def perturb_row(row, cols, variation=0.02):
    row_out = row.copy()
    original_total = row[cols].sum(skipna=True)

    #perturb each value ±2%
    for col in cols:
        val = row[col]
        if pd.notna(val):
            rand = np.random.uniform(1 - variation, 1 + variation)
            row_out[col] = val * rand

    new_total = row_out[cols].sum(skipna=True)

    #scale only if original total was near 100
    if 90 <= original_total <= 100:
        target_total = np.random.uniform(98.5, 100)
        scale = target_total / new_total
        for col in cols:
            if pd.notna(row_out[col]):
                row_out[col] *= scale

    return row_out

#apply perturbation row-wise
df_synthetic = df_expanded.apply(lambda row: perturb_row(row, composition_cols), axis=1)

#round off to 3 decimal places
df_synthetic[composition_cols] = df_synthetic[composition_cols].round(3)

# # Optional: Reset index and save to file
# df_synthetic = df_synthetic.reset_index(drop=True)
# df_synthetic.to_csv("actinolite_synthetic.csv", index=False)
